In [21]:
import os
import re
import json
import math
import argparse
import numpy as np
from pathlib import Path
from collections import defaultdict

import pandas as pd
import torch
import nltk

from nltk.tokenize import sent_tokenize
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer

In [ ]:
# clean keywords and filter out empty strings, masked keywords, and punctuation

def normalise_phrase(text):
    if text is None:
        return None

    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip(" \t\n\r.,;:!?\"'`“”‘’()[]{}")

    if not text or text == "<mask>":
        return None
    if re.fullmatch(r"[\W_]+", text):
        return None

    return text

In [23]:
# sort keywords by importance, choose words to mask

def select_keywords_to_mask(keyword_score_map, mask_fraction=1.0, min_keywords_to_mask=1):
    ranked = sorted(keyword_score_map.items(), key=lambda x: x[1], reverse=True)

    if not ranked or mask_fraction <= 0:
        return []

    n_total = len(ranked)
    n_mask = int(math.ceil(n_total * mask_fraction))
    n_mask = max(min_keywords_to_mask, n_mask)

    return [kw for kw, _ in ranked[:n_mask]]

In [24]:
def mask_keywords_in_text(text, keywords):
    if not text or not keywords:
        return text

    sorted_keywords = sorted(
        set(keywords),
        key=lambda x: (len(x.split()), len(x)),
        reverse=True
    )

    for kw in sorted_keywords:
        pattern = re.compile(rf"(?<!\w){re.escape(kw)}(?!\w)", flags=re.IGNORECASE)
        text = pattern.sub("<MASK>", text)

    return text

In [25]:
# split long strings into chunks based on the tokenizer lenght

def chunk_text_by_tokens(text, tokenizer, max_tokens, overlap_tokens=20):
    text = text.strip()
    if not text:
        return []

    enc = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True,
    )

    offsets = enc["offset_mapping"]

    if len(offsets) <= max_tokens:
        return [text]

    chunks = []
    start_token = 0

    while start_token < len(offsets):
        end_token = min(start_token + max_tokens, len(offsets))

        start_char = offsets[start_token][0]
        end_char = offsets[end_token - 1][1]

        chunk = text[start_char:end_char].strip()
        if chunk:
            chunks.append(chunk)

        if end_token >= len(offsets):
            break

        start_token = max(0, end_token - overlap_tokens)

    return chunks

In [26]:
# run KeyBERT on batches of text chunks

def extract_keywords_batched(
    kw_model,
    docs,
    batch_size=128,
    top_n=10,
    ngram_range=(1, 2),
    stop_words="english",
):
    all_results = []

    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]

        results = kw_model.extract_keywords(
            batch,
            keyphrase_ngram_range=ngram_range,
            stop_words=stop_words,
            top_n=top_n,
        )

        if len(batch) == 1 and isinstance(results[0], tuple):
            results = [results]

        all_results.extend(results)

    return all_results

In [ ]:
def process_single_article(
    text,
    kw_model,
    tokenizer,
    max_chunk_tokens,
    overlap_tokens,
    top_k_per_sentence,
    mask_fraction,
):
    sentences = [s.strip() for s in sent_tokenize(text) if s.strip()]
    sentence_scores = [defaultdict(float) for _ in sentences]

    all_chunks = []
    chunk_map = []

    for i, sentence in enumerate(sentences):
        chunks = chunk_text_by_tokens(sentence, tokenizer, max_chunk_tokens, overlap_tokens)
        for chunk in chunks:
            all_chunks.append(chunk)
            chunk_map.append(i)

    if all_chunks:
        results = extract_keywords_batched(
            kw_model,
            all_chunks,
            top_n=max(8, top_k_per_sentence * 4),
        )

        for (sent_idx, chunk_res) in zip(chunk_map, results):
            for phrase, score in chunk_res:
                phrase = normalise_phrase(phrase)
                if not phrase:
                    continue

                if phrase not in sentence_scores[sent_idx] or score > sentence_scores[sent_idx][phrase]:
                    sentence_scores[sent_idx][phrase] = score

    # Aggregate
    article_scores = {}
    sentence_keywords = []

    for sent_map in sentence_scores:
        ranked = sorted(sent_map.items(), key=lambda x: x[1], reverse=True)
        kws = [kw for kw, _ in ranked[:top_k_per_sentence]]
        sentence_keywords.append(kws)

        for kw, score in ranked:
            if kw not in article_scores or score > article_scores[kw]:
                article_scores[kw] = score

    all_keywords = sorted(article_scores, key=article_scores.get, reverse=True)

    masked_keywords = select_keywords_to_mask(article_scores, mask_fraction)
    masked_text = mask_keywords_in_text(text, masked_keywords)

    return {
        "masked_text": masked_text,
        "keywords": all_keywords,
        "masked_keywords": masked_keywords,
        "sentence_keywords": sentence_keywords,
    }

In [ ]:
''' Pipeline for one article:
- split text into sentences (NLTK)
- chunk each sentence into token-sized pieces
- extract keywords per chunk (KeyBERT)
- aggregate: per sentence/article
- rank keywords
- select keywords to mask
- mask them in the original text '''

def run_pipeline(
    df,
    text_col_idx="Body",                 # your text column
    title_col="Title",
    category_col="Category",
    output_path="SciDCC-masked-0.75.jsonl",
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    max_seq_length=None,
    top_k_per_sentence=3,
    mask_fraction=None,
    fsync_every=10,
):
    ## models
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(model_name, device=device)

    if max_seq_length:
        model.max_seq_length = max_seq_length

    tokenizer = model.tokenizer
    kw_model = KeyBERT(model)
    max_chunk_tokens = model.max_seq_length - 2

    ## output
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    processed_counter = 0

    with open(output_path, "a", encoding="utf-8") as fh:

        for idx, row in df.iterrows():
            # support column name or index
            if isinstance(text_col_idx, int):
                value = row.iloc[text_col_idx]
            else:
                value = row[text_col_idx]

            text = "" if pd.isna(value) else str(value)

            result = process_single_article(
                text=text,
                kw_model=kw_model,
                tokenizer=tokenizer,
                max_chunk_tokens=max_chunk_tokens,
                overlap_tokens=20,
                top_k_per_sentence=top_k_per_sentence,
                mask_fraction=mask_fraction,
            )

            record = {
                "Title": row.get(title_col, None),
                "Category": row.get(category_col, None),
                "masked_text": result["masked_text"],
            }

            fh.write(json.dumps(record, ensure_ascii=False) + "\n")
            fh.flush()

            processed_counter += 1
            if processed_counter % fsync_every == 0:
                os.fsync(fh.fileno())

            print(f"Saved row {idx}")

    print("All complete, woohooo!")

In [ ]:
data = pd.read_csv("../../data/scidcc/SciDCC.csv")

df = data.sample(n=1000, random_state=16).reset_index(drop=True)

run_pipeline(
    df,
    mask_fraction=0.75
)